# Обслуживание таблиц: компакция, очистка снапшотов и orphan-файлов

Iceberg-таблица, в которую часто пишут маленькими порциями (частые `INSERT`,
`UPDATE`, `MERGE`, потоковая запись), со временем накапливает:

1. **много мелких data-файлов** — это замедляет чтение (больше файлов =
   больше overhead на открытие/планирование);
2. **много снапшотов** — каждый снапшот держит ссылки на manifest-файлы и не
   даёт удалить старые данные, даже если они больше никому не нужны;
3. **orphan-файлы** — файлы, оставшиеся в S3 после упавших/прерванных задач и
   не привязанные ни к одному снапшоту.

Для решения этих проблем в Iceberg есть встроенные "процедуры обслуживания",
вызываемые через `CALL <catalog>.system.<procedure>(...)`.

In [1]:
from pyspark.sql import SparkSession

# Настраиваем каталог Iceberg прямо в коде, а не через spark-defaults.conf.
# Это даёт тот же результат, но позволяет менять параметры каталога
# независимо от конфигурации образа и делает ноутбук самодостаточным.
spark = (
    SparkSession.builder
    .appName("Table maintenance")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.course", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.course.type", "rest")
    .config("spark.sql.catalog.course.uri", "http://rest-catalog:8181")
    .config("spark.sql.catalog.course.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.course.warehouse", "s3://warehouse")
    .config("spark.sql.catalog.course.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.course.s3.path-style-access", "true")
    .getOrCreate()
)

print("Spark + Iceberg готовы к работе")

Spark + Iceberg готовы к работе


Ниже определена вспомогательная функция `read_metadata_table` для чтения
служебных metadata-таблиц Iceberg в обход REST-каталога этого стенда —
подробное объяснение, зачем она нужна, см. в `00_Setup_and_Basics.ipynb`.
Вызовы `CALL course.system....` эту проблему не затрагивают и работают как
обычно — namespace `system` у них всегда одноуровневый.

In [2]:
from pyspark.sql import DataFrame


def read_metadata_table(spark, table_identifier, metadata_type):
    """Читает metadata-таблицу Iceberg (history, snapshots, files, partitions, manifests, ...)
    в обход REST-каталога, который отклоняет составные namespace в URL
    (см. подробное объяснение в 00_Setup_and_Basics.ipynb)."""
    jvm = spark._jvm
    table = jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, table_identifier)
    mtype = jvm.org.apache.iceberg.MetadataTableType.valueOf(metadata_type.upper())
    jdf = jvm.org.apache.iceberg.spark.SparkTableUtil.loadMetadataTable(spark._jsparkSession, table, mtype)
    return DataFrame(jdf, spark)

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS course.test")

spark.sql("DROP TABLE IF EXISTS course.test.maintenance_demo")
spark.sql("""
CREATE TABLE course.test.maintenance_demo (
    id   BIGINT,
    name STRING
)
USING iceberg
""")

# Пишем много маленьких коммитов, чтобы получить кучу мелких файлов и снапшотов —
# именно так выглядит таблица после недели частых стриминговых/батчевых записей.
for i in range(10):
    spark.sql(f"INSERT INTO course.test.maintenance_demo VALUES ({i}, 'row_{i}')")

print("Файлов данных:", read_metadata_table(spark, "course.test.maintenance_demo", "files").count())
print("Снапшотов:", read_metadata_table(spark, "course.test.maintenance_demo", "snapshots").count())

Файлов данных: 10
Снапшотов: 10


## Компакция: `rewrite_data_files`

Склеивает мелкие data-файлы в более крупные (по умолчанию целится в файлы
~512 МБ), что уменьшает их количество и ускоряет последующее чтение.
Старые файлы физически не удаляются сразу — они просто перестают быть частью
*текущего* снапшота (их подчистит `expire_snapshots`, см. ниже).

In [4]:
result = spark.sql("CALL course.system.rewrite_data_files(table => 'test.maintenance_demo')")
result.show(truncate=False)

print("Файлов данных после компакции:", read_metadata_table(spark, "course.test.maintenance_demo", "files").count())

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|10                        |1                     |6688                 |0                      |0                         |
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+

Файлов данных после компакции: 1


## Очистка старых снапшотов: `expire_snapshots`

Компакция создала ещё один снапшот, а старые 10 снапшотов (по одному на
`INSERT`) всё ещё числятся в metadata и не дают собирать мусор. `expire_snapshots`
удаляет снапшоты старше указанного порога (кроме последних `retain_last`),
после чего файлы, на которые никто больше не ссылается, физически удаляются
из S3.

В проде это обычно вызывают по расписанию (например, раз в сутки, оставляя
снапшоты за последнюю неделю), а не сразу после каждой записи — снапшоты
нужны и для time travel, и для параллельно идущих long-running запросов.

In [5]:
from datetime import datetime

print("Снапшотов до очистки:", read_metadata_table(spark, "course.test.maintenance_demo", "snapshots").count())

# CALL-процедуры Iceberg принимают timestamp только как литерал TIMESTAMP \'...\',
# поэтому текущее время считаем на стороне Python и подставляем строкой.
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
spark.sql(f"""
CALL course.system.expire_snapshots(
    table => 'test.maintenance_demo',
    older_than => TIMESTAMP '{now}',
    retain_last => 1
)
""").show(truncate=False)

print("Снапшотов после очистки:", read_metadata_table(spark, "course.test.maintenance_demo", "snapshots").count())

Снапшотов до очистки: 11
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|10                      |0                                  |0                                  |10                          |10                          |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+

Снапшотов после очистки: 1


## Удаление файлов-сирот: `remove_orphan_files`

Находит файлы в директории таблицы на S3, которые не упомянуты ни в одном
manifest-файле ни одного снапшота (обычно — мусор от упавших джобов), и
удаляет те, что старше заданного порога (по умолчанию — старше 3 дней, чтобы
не зацепить файлы от ещё не закоммиченной параллельно идущей записи).

Порог не может быть меньше 24 часов от текущего момента — это встроенная
защита от случайного удаления файлов активной, ещё не закоммиченной записи.
В нашей только что созданной демо-таблице настоящих orphan-файлов нет и
старше суток тоже ничего нет, поэтому вызов ниже корректно найдёт 0 файлов —
именно так он и должен вести себя на свежей таблице.

In [6]:
from datetime import timedelta

older_than = (datetime.now() - timedelta(hours=25)).strftime("%Y-%m-%d %H:%M:%S.%f")
spark.sql(f"""
CALL course.system.remove_orphan_files(
    table => 'test.maintenance_demo',
    older_than => TIMESTAMP '{older_than}'
)
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------+
|orphan_file_location                                                                                   |
+-------------------------------------------------------------------------------------------------------+
|s3://warehouse/test/maintenance_demo/data/00000-0-6859178e-f86e-4c81-b619-43a8293825fa-0-00001.parquet |
|s3://warehouse/test/maintenance_demo/data/00000-0-c112b110-c456-4da9-a31a-47af36644e02-0-00001.parquet |
|s3://warehouse/test/maintenance_demo/data/00000-0-fd963c65-95cf-453e-ae65-bd864e6aa640-0-00001.parquet |
|s3://warehouse/test/maintenance_demo/data/00000-1-820173d2-eee1-446a-b26c-54cfb21eaa62-0-00001.parquet |
|s3://warehouse/test/maintenance_demo/data/00000-1-d62786cf-1563-4c82-9f8a-a767b5b348d9-0-00001.parquet |
|s3://warehouse/test/maintenance_demo/data/00000-1-ddaf7797-a93b-449f-bd0c-388045162a05-0-00001.parquet |
|s3://warehouse/test/maintenance_demo/data/000

## Консолидация манифестов: `rewrite_manifests`

Каждый коммит по умолчанию добавляет свой manifest-файл. Если коммитов было
много (как в нашем примере — 10 инсертов), список манифестов, которые нужно
прочитать при планировании запроса, тоже разрастается. `rewrite_manifests`
объединяет их в более крупные, ускоряя planning-фазу запросов.

In [7]:
spark.sql("CALL course.system.rewrite_manifests(table => 'test.maintenance_demo')").show(truncate=False)

+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|11                       |1                    |
+-------------------------+---------------------+



## Итог: что и когда запускать

| Процедура              | Что решает                                   | Как часто               |
|-------------------------|-----------------------------------------------|--------------------------|
| `rewrite_data_files`    | Много мелких data-файлов                      | Регулярно (напр. ежедневно/еженедельно), после активной записи |
| `rewrite_manifests`     | Много мелких manifest-файлов, медленный planning | Вместе с компакцией данных |
| `expire_snapshots`      | Рост метаданных, недоступность GC старых файлов | По расписанию, с запасом retention для time travel |
| `remove_orphan_files`   | Мусорные файлы от упавших джобов              | Редко (напр. раз в неделю), с осторожным `older_than` |

Все четыре процедуры безопасно вызывать на "боевой" таблице во время чтения —
они не блокируют читателей и работают поверх snapshot isolation Iceberg.

In [8]:
spark.stop()